In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# First-Order Logic Parser (Shunting Yard)

This notebook implements a parser for first-order logic and arithmetic terms using the shunting yard algorithm.  The result of the parser is a value of a variant type that describes exactly what a term or a formula is.

### Terms and Formulas

The syntax of our formulas is as follows:
- *Variables* start with an **upper case** letter (e.g., `X`, `Y`).
- *Function symbols* and *predicate symbols* start with a **lower case** letter (e.g., `e`, `p`).  Numbers are constants, i.e. function symbols without arguments.
- The arithmetic operators `+`, `-`, `*`, `/`, `%`, and `**` can be used as infix operators.
- The relational operators `<`, `>`, `=`, `≤`, `≥` can be used as infix operators.
- The logical symbols `⊤`, `⊥`, `∧`, `∨`, `¬`, `→`, `↔`, `∀`, `∃` are supported with the usual precedences.  A quantified formula is written as `∀X: f` or `∃X: f`.

A *term* is either a variable or a function symbol applied to a list of terms.  The type `term` has a constructor for each case:
- `Var x` represents the variable `x`,
- `Fun (f, [t1; ...; tn])` represents the term $f(t_1, \cdots, t_n)$.  A constant `c` is represented as `Fun (c, [])`, and an arithmetic term like `X + 2` is represented as `Fun ("+", [Var "X"; Fun ("2", [])])`.

The type `formula` has a constructor for every kind of formula.  An atomic formula $p(t_1, \cdots, t_n)$ is represented as `Atom (p, [t1; ...; tn])`.  Equations and inequations are atomic formulas, too: `X = Y` is represented as `Atom ("=", [Var "X"; Var "Y"])`.  The quantified formula $\forall x: f$ is represented as `Forall (x, f)`.

In [ ]:
type term =
  | Var of string                  (* variable *)
  | Fun of string * term list      (* function application, constants have no arguments *)

type formula =
  | Verum                          (* ⊤ *)
  | Falsum                         (* ⊥ *)
  | Atom   of string * term list   (* atomic formula p(t1, ..., tn) *)
  | Not    of formula              (* ¬ *)
  | And    of formula * formula    (* ∧ *)
  | Or     of formula * formula    (* ∨ *)
  | Imp    of formula * formula    (* → *)
  | Iff    of formula * formula    (* ↔ *)
  | Forall of string * formula     (* ∀x: f *)
  | Exists of string * formula     (* ∃x: f *)

### Converting Terms and Formulas into Strings

Values of the types `term` and `formula` are hard to read.  Therefore, we define functions that convert them into strings.

The function `term_to_string` takes one argument:
- `t` is a term.

It returns a string representing `t`, e.g. `f(X, g(Y))`.  A constant is written without parentheses.  The arithmetic operators are written as infix operators, and every arithmetic term is enclosed in parentheses, so that its structure is unambiguous: The term `X + 2 * Y` is written as `(X + (2 * Y))`.

The list `arithmetic_ops` contains the arithmetic operators, the list `relational_ops` contains the relational operators.

In [ ]:
let arithmetic_ops : string list = ["+"; "-"; "*"; "/"; "%"; "**"]
let relational_ops : string list = ["="; "<"; ">"; "≤"; "≥"]

let rec term_to_string (t : term) : string =
  match t with
  | Var x -> x
  | Fun (c, []) -> c
  | Fun (op, [s; t]) when List.mem op arithmetic_ops ->
      "(" ^ term_to_string s ^ " " ^ op ^ " " ^ term_to_string t ^ ")"
  | Fun (f, ts) -> f ^ "(" ^ String.concat ", " (List.map term_to_string ts) ^ ")"

The function `formula_to_string` takes one argument:
- `f` is a formula.

It returns a string representing `f`.  Every binary subformula is enclosed in parentheses.  The relational operators are written as infix operators.

**Implementation:** A quantifier binds stronger than the binary operators.  For example, the parser reads `∀X: p(X) ∧ q` as `(∀X: p(X)) ∧ q`.  Therefore, if the left operand of a binary operator starts with a quantifier, possibly preceded by negations, this operand is enclosed in parentheses.  The local function `is_quantified` checks this condition.  A quantified formula that is the right operand extends to the closing parenthesis of the binary formula anyway.  Hence the parser reads the result of `formula_to_string` back as the same formula.

In [ ]:
let rec formula_to_string (f : formula) : string =
  let rec is_quantified g =
    match g with
    | Forall _ | Exists _ -> true
    | Not h               -> is_quantified h
    | _                   -> false
  in
  let binary op g h =
    let left = if is_quantified g then "(" ^ formula_to_string g ^ ")" else formula_to_string g in
    "(" ^ left ^ " " ^ op ^ " " ^ formula_to_string h ^ ")"
  in
  match f with
  | Verum  -> "⊤"
  | Falsum -> "⊥"
  | Atom (r, [s; t]) when List.mem r relational_ops ->
      term_to_string s ^ " " ^ r ^ " " ^ term_to_string t
  | Atom (p, ts)  -> term_to_string (Fun (p, ts))
  | Not g         -> "¬" ^ formula_to_string g
  | And (g, h)    -> binary "∧" g h
  | Or  (g, h)    -> binary "∨" g h
  | Imp (g, h)    -> binary "→" g h
  | Iff (g, h)    -> binary "↔" g h
  | Forall (x, g) -> "∀" ^ x ^ ": " ^ formula_to_string g
  | Exists (x, g) -> "∃" ^ x ^ ": " ^ formula_to_string g

In [ ]:
formula_to_string (Forall ("X", Imp (Atom ("p", [Var "X"]), Atom ("=", [Fun ("+", [Var "X"; Fun ("1", [])]); Fun ("f", [Var "X"])]))))

### Raw Syntax Trees

While the shunting yard algorithm is running, it does not yet know whether a part of the input is a term or a formula.  Therefore, the algorithm builds *raw syntax trees* of type `raw`:
- `Leaf s` is a variable or a number,
- `Node (op, args)` is an operator, a function symbol, or a predicate symbol `op` applied to the list `args` of raw syntax trees.  A quantified formula $\forall x: f$ is represented as `Node ("∀", [Leaf x; f])`.

Once the whole input has been parsed, the raw syntax tree is converted into a formula.  This conversion checks that every part of the tree is used correctly.

In [ ]:
type raw =
  | Leaf of string
  | Node of string * raw list

The list `logical_ops` contains the logical operators.  They can not be used as function or predicate symbols.

In [ ]:
let logical_ops : string list = ["⊤"; "⊥"; "¬"; "↔"; "→"; "∨"; "∧"; "∀"; "∃"]

The function `is_var` takes one argument:
- `s` is a string.

It returns `true` if `s` is a variable, i.e. if `s` starts with an upper case letter that is followed by letters, digits, or underscores.

The function `is_sym` takes one argument:
- `s` is a string.

It returns `true` if `s` is a function or predicate symbol, i.e. if `s` starts with a lower case letter that is followed by letters, digits, or underscores.

The function `is_num` takes one argument:
- `s` is a string.

It returns `true` if `s` is a number, i.e. a nonempty sequence of digits that may contain one decimal point.

In [ ]:
let is_upper (c : char) : bool = 'A' <= c && c <= 'Z'
let is_lower (c : char) : bool = 'a' <= c && c <= 'z'
let is_digit (c : char) : bool = '0' <= c && c <= '9'
let is_ident_char (c : char) : bool = is_upper c || is_lower c || is_digit c || c = '_'

let is_var (s : string) : bool =
  s <> "" && is_upper s.[0] && String.for_all is_ident_char s

let is_sym (s : string) : bool =
  s <> "" && is_lower s.[0] && String.for_all is_ident_char s

let is_num (s : string) : bool =
  s <> "" && is_digit s.[0] && String.for_all (fun c -> is_digit c || c = '.') s
  && List.length (String.split_on_char '.' s) <= 2

In [ ]:
is_var "X1", is_sym "e", is_num "3.14", is_num "1.2.3"

### The Tokenizer

The function `tokenize` takes one argument:
- `s` is a string that is supposed to represent a formula.

It returns the list of tokens of `s`.  A token is either an identifier (a variable or a symbol), a number, the operator `**`, or one of the characters
`⊤ ⊥ ∧ ∨ ¬ → ↔ ( ) ∀ ∃ : , < > = ≤ ≥ + - * / %`.  Blanks, tabs, and newlines are skipped.  If `s` contains any other character, an exception is raised.

**Implementation:** As in the parser for propositional logic, the string is scanned from left to right by the recursive function `scan`, where `scan i` returns the tokens of the part of `s` that starts at position `i`.  The function `span p j` returns the position of the first character at or after position `j` that does not satisfy the predicate `p`.  Some of the special characters occupy several bytes in UTF-8.  Therefore, `starts_at op i` checks whether the string `op` occurs in `s` at position `i`.  The operator `**` is tested before `*`, so that it is recognized as one token.

In [ ]:
let special_tokens : string list =
  ["**"; "⊤"; "⊥"; "∧"; "∨"; "¬"; "→"; "↔"; "("; ")"; "∀"; "∃"; ":"; ",";
   "<"; ">"; "="; "≤"; "≥"; "+"; "-"; "*"; "/"; "%"]

let tokenize (s : string) : string list =
  let n = String.length s in
  let rec span p j = if j < n && p s.[j] then span p (j + 1) else j in
  let starts_at op i =
    i + String.length op <= n && String.sub s i (String.length op) = op
  in
  let rec scan i =
    if i >= n then
      []
    else if s.[i] = ' ' || s.[i] = '\t' || s.[i] = '\n' then
      scan (i + 1)
    else if is_upper s.[i] || is_lower s.[i] then
      let j = span is_ident_char (i + 1) in
      String.sub s i (j - i) :: scan j
    else if is_digit s.[i] then
      let j = span (fun c -> is_digit c || c = '.') (i + 1) in
      String.sub s i (j - i) :: scan j
    else
      match List.find_opt (fun op -> starts_at op i) special_tokens with
      | Some op -> op :: scan (i + String.length op)
      | None    -> failwith (Printf.sprintf "tokenize: unexpected character at position %d of %s" i s)
  in
  scan 0

In [ ]:
tokenize "∀X: ∃Y: p(X, Y) → X ** 2 ≥ 0"

### Precedences

The function `is_prefix` takes one argument:
- `op` is an operator on the operator stack.

It returns `true` if `op` is a *prefix operator*, i.e. the negation `¬` or a quantifier.  On the operator stack, the quantifiers $\forall x$ and $\exists x$ are stored as the strings `"∀|x"` and `"∃|x"`, respectively.

The function `precedence` takes one argument:
- `op` is an operator.

It returns the precedence of `op`.  Operators with a higher precedence bind stronger.  The prefix operators bind stronger than `∧` and `∨`, but weaker than the relational operators.  Hence `¬X = Y` is parsed as `¬(X = Y)`, and `∀X: p(X) ∧ q(X)` is parsed as `(∀X: p(X)) ∧ q(X)`.  The TypeScript version uses the precedence 4.5 for the prefix operators.  We multiply all precedences by 2 so that they are integers.  Other strings, e.g. function symbols, have the precedence 0.

In [ ]:
let is_prefix (op : string) : bool =
  op = "¬" || String.starts_with ~prefix:"∀|" op || String.starts_with ~prefix:"∃|" op

let precedence (op : string) : int =
  if is_prefix op then 9
  else match op with
    | "↔" -> 2
    | "→" -> 4
    | "∨" -> 6
    | "∧" -> 8
    | "=" | "<" | ">" | "≤" | "≥" -> 10
    | "+" | "-" -> 12
    | "*" | "/" | "%" -> 14
    | "**" -> 16
    | _ -> 0

The function `eval_before` takes two arguments:
- `stack_op` is the operator on top of the operator stack,
- `next_op` is the next operator from the input.

It returns `true` if `stack_op` has to be evaluated before `next_op` is processed.  This is the case if `stack_op` has a higher precedence than `next_op`, or if both operators have the same precedence and `next_op` is left associative.  The operators `**` and `→` are right associative.  The operator `↔` is not associative, i.e. a string like `A ↔ B ↔ C` is rejected.  Two prefix operators are evaluated from right to left: In `¬¬p`, the second negation is applied first.  An opening parenthesis is never evaluated.

In [ ]:
let eval_before (stack_op : string) (next_op : string) : bool =
  if stack_op = "(" then
    false
  else
    let prec_stack = precedence stack_op in
    let prec_next  = precedence next_op in
    if prec_stack > prec_next then
      true
    else if prec_stack = prec_next then begin
      if next_op = "**" || next_op = "→" then false
      else if stack_op = "↔" && next_op = "↔" then failwith "↔ is not associative"
      else if is_prefix stack_op && is_prefix next_op then false
      else true
    end else
      false

### The Shunting Yard Algorithm

The argument stack contains raw syntax trees and *markers*.  A marker is pushed whenever an opening parenthesis is read.  When the matching closing parenthesis is read, the arguments above the marker are the arguments of a function or predicate symbol, or the single expression enclosed in the parentheses.

In [ ]:
type item =
  | Raw of raw
  | Marker

The function `parse_raw` takes one argument:
- `s` is a string representing a formula.

It returns the raw syntax tree of `s`.  If `s` can not be parsed, an exception is raised.

**Implementation:** The algorithm uses three mutable stacks, which are references to lists: `tokens` holds the tokens that still have to be processed, `operators` is the operator stack, and `arguments` is the argument stack.  The local function `pop_argument` removes the top of the argument stack and returns it.  The function `pop_and_evaluate` removes the operator on top of the operator stack and applies it to its arguments.  The tokens are processed in a loop:
1. Variables and numbers are pushed onto the argument stack.
2. The constants `⊤` and `⊥` are pushed onto the argument stack.
3. A function or predicate symbol that is followed by an opening parenthesis is pushed onto the operator stack.  Otherwise, it is a constant and pushed onto the argument stack.
4. A quantifier `∀X:` or `∃X:` is pushed onto the operator stack as `"∀|X"` or `"∃|X"`.
5. If the operator stack is empty or the token is an opening parenthesis, the token is pushed onto the operator stack.  For an opening parenthesis, a marker is pushed onto the argument stack.
6. For a closing parenthesis, all operators up to the matching opening parenthesis are evaluated and the opening parenthesis is removed.  If there is a function or predicate symbol below the parenthesis, the arguments above the marker become its arguments.  Otherwise, the parentheses just group an expression.
7. For a comma, which separates arguments, all operators up to the opening parenthesis are evaluated.
8. Otherwise, the token is an infix operator.  If the operator on top of the stack has to be evaluated first, it is evaluated and the token is put back.  Otherwise, the token is pushed onto the operator stack.

Finally, all remaining operators are evaluated.  If the input is well formed, the argument stack contains exactly one raw syntax tree.

In [ ]:
let parse_raw (s : string) : raw =
  let tokens    = ref (tokenize s) in
  let operators = ref [] in
  let arguments = ref [] in
  let next () =
    match !tokens with
    | t :: rest -> tokens := rest; t
    | []        -> failwith ("unexpected end of input in " ^ s)
  in
  let push_op op = operators := op :: !operators in
  let push_arg a = arguments := a :: !arguments in
  let top_op () = match !operators with op :: _ -> Some op | [] -> None in
  let pop_op () =
    match !operators with
    | op :: rest -> operators := rest; op
    | []         -> failwith ("unexpected end of input in " ^ s)
  in
  let pop_argument () =
    match !arguments with
    | Raw a :: rest -> arguments := rest; a
    | _             -> failwith ("missing argument in " ^ s)
  in
  let pop_and_evaluate () =
    let op = pop_op () in
    if op = "¬" then
      push_arg (Raw (Node ("¬", [pop_argument ()])))
    else if is_prefix op then begin
      (* op has the form "∀|X" or "∃|X"; the quantifier occupies 3 bytes *)
      let q = String.sub op 0 3 in
      let x = String.sub op 4 (String.length op - 4) in
      push_arg (Raw (Node (q, [Leaf x; pop_argument ()])))
    end else if List.mem op ["↔"; "→"; "∨"; "∧"; "="; "<"; ">"; "≤"; "≥"; "+"; "-"; "*"; "/"; "%"; "**"] then begin
      let rhs = pop_argument () in
      let lhs = pop_argument () in
      push_arg (Raw (Node (op, [lhs; rhs])))
    end else
      failwith (Printf.sprintf "unknown operator %s in %s" op s)
  in
  while !tokens <> [] do
    let next_op = next () in
    if is_var next_op || is_num next_op then                     (* 1 *)
      push_arg (Raw (Leaf next_op))
    else if next_op = "⊤" || next_op = "⊥" then                  (* 2 *)
      push_arg (Raw (Node (next_op, [])))
    else if is_sym next_op then begin                            (* 3 *)
      match !tokens with
      | "(" :: _ -> push_op next_op
      | _        -> push_arg (Raw (Node (next_op, [])))
    end else if next_op = "∀" || next_op = "∃" then begin        (* 4 *)
      let x = next () in
      if not (is_var x) then failwith ("expected a variable after " ^ next_op ^ " in " ^ s);
      if next () <> ":" then failwith ("expected ':' after " ^ x ^ " in " ^ s);
      push_op (next_op ^ "|" ^ x)
    end else if !operators = [] || next_op = "(" then begin      (* 5 *)
      push_op next_op;
      if next_op = "(" then push_arg Marker
    end else if next_op = ")" then begin                         (* 6 *)
      while top_op () <> None && top_op () <> Some "(" do pop_and_evaluate () done;
      if top_op () = None then failwith ("mismatched parentheses in " ^ s);
      ignore (pop_op ());
      match top_op () with
      | Some f when is_sym f ->
          ignore (pop_op ());
          let rec collect args =
            match !arguments with
            | Marker :: rest -> arguments := rest; args
            | Raw a  :: rest -> arguments := rest; collect (a :: args)
            | []             -> failwith ("missing marker in " ^ s)
          in
          push_arg (Raw (Node (f, collect [])))
      | _ ->
          let result = pop_argument () in
          (match !arguments with
           | Marker :: rest -> arguments := rest
           | _              -> failwith ("empty parentheses in " ^ s));
          push_arg (Raw result)
    end else if next_op = "," then begin                         (* 7 *)
      while top_op () <> None && top_op () <> Some "(" do pop_and_evaluate () done
    end else begin                                               (* 8 *)
      match top_op () with
      | Some stack_op when eval_before stack_op next_op ->
          pop_and_evaluate ();
          tokens := next_op :: !tokens
      | _ -> push_op next_op
    end
  done;
  while !operators <> [] do pop_and_evaluate () done;
  match !arguments with
  | [Raw result] -> result
  | _            -> failwith ("could not parse " ^ s)

In [ ]:
parse_raw "∀X: e * X = X"

### From Raw Syntax Trees to Terms and Formulas

The function `to_term` takes one argument:
- `r` is a raw syntax tree.

It converts `r` into a term.  A leaf is a variable if it starts with an upper case letter, otherwise it is a number, which is represented as a constant.  If `r` contains a logical operator, it is not a term and an exception is raised.

The function `to_formula` takes one argument:
- `r` is a raw syntax tree.

It converts `r` into a formula.  Every node whose operator is not a logical operator is an atomic formula, whose arguments have to be terms.  If `r` is not a formula, an exception is raised.

**Implementation:** `List.map to_term args` converts the arguments of a function or predicate symbol.  The patterns check the number of arguments of the logical operators, e.g. the pattern `Node ("¬", [f])` only matches a negation with exactly one argument.

In [ ]:
let rec to_term (r : raw) : term =
  match r with
  | Leaf s when is_var s -> Var s
  | Leaf s -> Fun (s, [])
  | Node (f, args) when not (List.mem f logical_ops) -> Fun (f, List.map to_term args)
  | Node (op, _) -> failwith ("to_term: the operator " ^ op ^ " can not occur in a term")

let rec to_formula (r : raw) : formula =
  match r with
  | Node ("⊤", [])          -> Verum
  | Node ("⊥", [])          -> Falsum
  | Node ("¬", [f])         -> Not (to_formula f)
  | Node ("∧", [f; g])      -> And (to_formula f, to_formula g)
  | Node ("∨", [f; g])      -> Or  (to_formula f, to_formula g)
  | Node ("→", [f; g])      -> Imp (to_formula f, to_formula g)
  | Node ("↔", [f; g])      -> Iff (to_formula f, to_formula g)
  | Node ("∀", [Leaf x; f]) -> Forall (x, to_formula f)
  | Node ("∃", [Leaf x; f]) -> Exists (x, to_formula f)
  | Node (p, args) when not (List.mem p logical_ops) -> Atom (p, List.map to_term args)
  | _ -> failwith "to_formula: this is not a formula"

The function `parse` takes one argument:
- `s` is a string representing a formula.

It returns the formula represented by `s`.  If `s` is not a well formed formula, an exception is raised.

In [ ]:
let parse (s : string) : formula =
  to_formula (parse_raw s)

The function `parse_term` takes one argument:
- `s` is a string representing a term.

It returns the term represented by `s`.  If `s` is not a well formed term, e.g. if it contains a logical operator, an exception is raised.

In [ ]:
let parse_term (s : string) : term =
  to_term (parse_raw s)

### Tests

In [ ]:
parse "∀X: e * X = X"

In [ ]:
parse "∀X: ∃Y: p(X, Y) → ∃Y: ∀X: p(X, Y)"

In [ ]:
parse "¬∀X: (red(X) → happy(X))"

In [ ]:
parse "X + 2 * Y = f(Z) ** 3"

In [ ]:
parse_term "f(g(X), Y)"

The function `formula_to_string` encloses quantified left operands in parentheses:

In [ ]:
formula_to_string (parse "∀X: ∃Y: p(X, Y) → ∃Y: ∀X: p(X, Y)")

The parser reads the strings produced by `formula_to_string` back as the same formulas:

In [ ]:
List.for_all (fun s -> let f = parse s in parse (formula_to_string f) = f)
  ["∀X: e * X = X"; "∀X: ∃Y: p(X, Y) → ∃Y: ∀X: p(X, Y)"; "¬∀X: (red(X) → happy(X))";
   "X + 2 * Y = f(Z) ** 3"; "(∀X: p(X)) ∧ q ∨ ¬∃Y: r(Y) ↔ ⊤"]

Strings that are not well formed formulas are rejected.  The following cells raise exceptions on purpose.

In [ ]:
parse "p ↔ q ↔ r"

In [ ]:
parse "f(X ∧ Y)"